In [24]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os 
%matplotlib qt

plt.style.use('seaborn-v0_8-whitegrid')  
plt.rcParams['font.family'] = 'Arial' 

output_folder = "foa_histograms"
os.makedirs(output_folder, exist_ok=True)

In [ ]:
### Read raw input data files ###

# Read the GR Parquet file (in UTC)
df_GR = pd.read_parquet('./datasets/Summer 2025 analysis/From GR/FORM_inference_2025july.pq')
locations = pd.read_parquet('./datasets/Summer 2025 analysis/From GR/FORM_new_station_locations.pq')
locations['short_name'] = ['GRS-BAE M7', 'GRS-BAE M14', 'GRS-BAE M31', 'GRS-BAE M54', 'HHN-HHO M14', 'HHN-HHO M28', 'HHN-HHO M44', 'HHN-HHO M59', 'HHN-HHO M80', 'HHN-HHO M91', 'SM-DDS M9', 'SM-DDS M29', 'SM-DDS M54', 'SM-DDS M78']
df_GR['Datetime'] = pd.to_datetime(df_GR['time'])

# Read the FOA .csv file (in UTC)
df_FOA_GRS_BAE = pd.read_csv('./datasets/Summer 2025 analysis/FOAs/GRS-BAE.csv')
df_FOA_DD_SM = pd.read_csv('./datasets/Summer 2025 analysis/FOAs/DD-SM.csv')
df_FOA_HHN_HHO = pd.read_csv('./datasets/Summer 2025 analysis/FOAs/HHN-HHO.csv')

In [ ]:
### Take hourly average of FOA data ###

hourly_avg_wind_speed = {}
foa_dfs = [
    ('df_FOA_GRS_BAE', df_FOA_GRS_BAE, ['wind_m7', 'wind_m14', 'wind_m31', 'wind_m54'], 'hourly_avg_GRS_BAE'),
    ('df_FOA_DD_SM', df_FOA_DD_SM, ['Wind M9', 'Wind M29', 'Wind M54', 'Wind M78'], 'hourly_avg_DD_SM'),
    ('df_FOA_HHN_HHO', df_FOA_HHN_HHO, ['Wind M14', 'Wind M28', 'Wind M44', 'Wind M59', 'Wind M80', 'Wind M91'], 'hourly_avg_HHN_HHO'),
]

for name, df, wind_cols, out_name in foa_dfs:

    # Set timestamp as index
    df['Time'] = pd.to_datetime(df['Time'])
    df.set_index('Time', inplace=True)

    # Clean text from data columns (each entry has 'm/s')
    for col in wind_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(r'\s*m/s', '', regex=True)
            .astype(float)
        )
    # Resample all wind columns at once
    hourly_avg = df[wind_cols].resample('h', label='left', closed='left').mean().reset_index()
    hourly_avg_wind_speed[out_name] = hourly_avg

# Assign to standalone variables
hourly_avg_GRS_BAE = hourly_avg_wind_speed['hourly_avg_GRS_BAE']
hourly_avg_DD_SM = hourly_avg_wind_speed['hourly_avg_DD_SM']
hourly_avg_HHN_HHO = hourly_avg_wind_speed['hourly_avg_HHN_HHO']

In [10]:
### Mapping of site_id to column names in FOA datasets ###
col_to_siteid_GRS_BAE = {
    'wind_m7': 111115,
    'wind_m14': 111116,
    'wind_m31': 111117,
    'wind_m54': 111118,
}
col_to_siteid_HHN_HHO = {
    'Wind M14': 111119,
    'Wind M28': 111120,
    'Wind M44': 111121,
    'Wind M59': 111122,
    'Wind M80': 111123,
    'Wind M91': 111124
}
col_to_siteid_DD_SM = {
    'Wind M9': 111125,
    'Wind M29': 111126,
    'Wind M54': 111127,
    'Wind M78': 111128
}

foa_configs = [
    # (name, col_to_siteid_dict, hourly_avg_df)
    ("GRS-BAE", col_to_siteid_GRS_BAE, hourly_avg_GRS_BAE),
    ("DD-SM", col_to_siteid_DD_SM, hourly_avg_DD_SM),
    ("HHN-HHO", col_to_siteid_HHN_HHO, hourly_avg_HHN_HHO),
]

In [28]:
### Make a histogram for wind speed differences (GR and FOAs) ###

# Set buckets for histogram (difference of wind speeds b/w -4 and 4 m/s with spacing of 1 m/s)
bin_edges = np.arange(-4, 5, 1)

for foa_name, col_to_siteid, hourly_avg_df in foa_configs:
    for foa_col, site_id in col_to_siteid.items():
        # Extract relevant data based on site
        df_gr_site = df_GR[df_GR['site_id'] == site_id][['Datetime', 'wspeed_pred']]
        df_foa_site = hourly_avg_df[['Time', foa_col]].rename(columns={foa_col: 'foa_wind'})
        
        # Merge data based on timestamp
        merged = pd.merge(
            df_gr_site, df_foa_site,
            left_on='Datetime', right_on='Time',
            how='inner',
            indicator=True
        )
        
        # Check for perfect merge
        n_gr = len(df_gr_site)
        n_foa = len(df_foa_site)
        n_merged = len(merged)
        if n_merged < min(n_gr, n_foa):
            print(f"WARNING: Imperfect merge for site_id {site_id} ({foa_col}) in {foa_name}")
            print(f"  Rows in GR dataset: {n_gr}, Rows in FOA dataset: {n_foa}, Rows successfully merged: {n_merged}\n")
            unmatched_gr = set(df_gr_site['Datetime']) - set(merged['Datetime'])
            unmatched_foa = set(df_foa_site['Time']) - set(merged['Time'])
        else:
            print(f"Perfect merge for site_id {site_id} ({foa_col}) in {foa_name}")
        
        # Compute the difference (FOA - GR)
        diff = merged['foa_wind'] - merged['wspeed_pred']
        counts, bins = np.histogram(diff, bins=bin_edges)
        total_count = len(diff)
        percentages = (counts/total_count)*100

        plt.figure(figsize=(8,6))
        plt.bar(bin_edges[:-1], percentages, width=np.diff(bin_edges)[0], align='edge', alpha=1, edgecolor='white', color=(0, 0.4470, 0.7410))
        plt.title(f"{foa_name} | Site ID: {site_id} ({foa_col})", fontsize=24, weight='bold')
        plt.xlabel("FOA - GR (m/s)", fontsize=18, weight='bold')
        plt.ylabel("Occurence (%)", fontsize=18, weight='bold')
        plt.ylim((0, 40))
        plt.gca().yaxis.grid(True, alpha=0.7)   
        plt.gca().xaxis.grid(False)    
        plt.gca().tick_params(axis='both', which='major', labelsize=14)   


        # Compute and display descriptive statistics
        mean_val = np.mean(diff)
        median_val = np.median(diff)
        std_val = np.std(diff)
        stats_text = (
            f"Mean discrepancy: {mean_val:.2f} m/s\n"
            f"Median discrepancy: {median_val:.2f} m/s\n"
            f"σ of discrepancies: {std_val:.2f} m/s"
        )
        plt.gca().text(
            0.96, 0.96, stats_text,
            transform=plt.gca().transAxes,
            ha='right', va='top',
            bbox=dict(facecolor='white', edgecolor='white', boxstyle='round,pad=0.3'),
            fontsize=14)

        filename = f"GR_{foa_name}_site{site_id}_{foa_col}.png"
        filepath = os.path.join(output_folder, filename)
        plt.savefig(filepath, bbox_inches='tight', dpi=150)
        plt.show()

  Rows in GR dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merged: 2024

  Rows in GR dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merged: 2024

  Rows in GR dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merged: 2024

  Rows in GR dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merged: 2024

  Rows in GR dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merged: 2024

  Rows in GR dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merged: 2024

  Rows in GR dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merged: 2024

Perfect merge for site_id 111128 (Wind M78) in DD-SM
  Rows in GR dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merged: 2024

  Rows in GR dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merged: 2024

  Rows in GR dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merged: 2024

  Rows in GR dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merg

In [27]:
### Make a histogram for wind speed differences (NWP and FOAs) ###

# Set buckets for histogram (difference of wind speeds b/w -4 and 4 m/s with spacing of 1 m/s)
bin_edges = np.arange(-4, 5, 1)

for foa_name, col_to_siteid, hourly_avg_df in foa_configs:
    for foa_col, site_id in col_to_siteid.items():
        # Extract relevant data based on site
        df_nwp_site = df_GR[df_GR['site_id'] == site_id][['Datetime', 'wspeed_nwp']]
        df_foa_site = hourly_avg_df[['Time', foa_col]].rename(columns={foa_col: 'foa_wind'})
        
        # Merge data based on timestamp
        merged = pd.merge(
            df_nwp_site, df_foa_site,
            left_on='Datetime', right_on='Time',
            how='inner',
            indicator=True
        )
        
        # Check for perfect merge
        n_nwp = len(df_nwp_site)
        n_foa = len(df_foa_site)
        n_merged = len(merged)
        if n_merged < min(n_nwp, n_foa):
            print(f"WARNING: Imperfect merge for site_id {site_id} ({foa_col}) in {foa_name}")
            print(f"  Rows in NWP dataset: {n_nwp}, Rows in FOA dataset: {n_foa}, Rows successfully merged: {n_merged}\n")
            unmatched_nwp = set(df_nwp_site['Datetime']) - set(merged['Datetime'])
            unmatched_foa = set(df_foa_site['Time']) - set(merged['Time'])
        else:
            print(f"Perfect merge for site_id {site_id} ({foa_col}) in {foa_name}")
        
        # Compute the difference (FOA - GR)
        diff = merged['foa_wind'] - merged['wspeed_nwp']
        counts, bins = np.histogram(diff, bins=bin_edges)
        total_count = len(diff)
        percentages = (counts/total_count)*100

        plt.figure(figsize=(8,6))
        plt.bar(bin_edges[:-1], percentages, width=np.diff(bin_edges)[0], align='edge', alpha=1, edgecolor='white', color=(0.8500, 0.3250, 0.0980))
        plt.title(f"{foa_name} | Site ID: {site_id} ({foa_col})", fontsize=24, weight='bold')
        plt.xlabel("FOA - NWP (m/s)", fontsize=18, weight='bold')
        plt.ylabel("Occurence (%)", fontsize=18, weight='bold')
        plt.ylim((0, 40))
        plt.gca().yaxis.grid(True, alpha=0.7)   
        plt.gca().xaxis.grid(False)    
        plt.gca().tick_params(axis='both', which='major', labelsize=14)   


        # Compute and display descriptive statistics
        mean_val = np.mean(diff)
        median_val = np.median(diff)
        std_val = np.std(diff)
        stats_text = (
            f"Mean discrepancy: {mean_val:.2f} m/s\n"
            f"Median discrepancy: {median_val:.2f} m/s\n"
            f"σ of discrepancies: {std_val:.2f} m/s"
        )
        plt.gca().text(
            0.96, 0.96, stats_text,
            transform=plt.gca().transAxes,
            ha='right', va='top',
            fontsize=14)
        
        filename = f"NWP_{foa_name}_site{site_id}_{foa_col}.png"
        filepath = os.path.join(output_folder, filename)
        plt.savefig(filepath, bbox_inches='tight', dpi=150)
        plt.show()

  Rows in NWP dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merged: 2024

  Rows in NWP dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merged: 2024

  Rows in NWP dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merged: 2024

  Rows in NWP dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merged: 2024

  Rows in NWP dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merged: 2024

  Rows in NWP dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merged: 2024

  Rows in NWP dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merged: 2024

Perfect merge for site_id 111128 (Wind M78) in DD-SM
  Rows in NWP dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merged: 2024

  Rows in NWP dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merged: 2024

  Rows in NWP dataset: 2025, Rows in FOA dataset: 2111, Rows successfully merged: 2024

  Rows in NWP dataset: 2025, Rows in FOA dataset: 2111, Rows succes

In [ ]:
### Make a histogram for wind speed differences (GR and NWP) ###

# Set MATLAB style
plt.style.use('seaborn-v0_8-whitegrid')  # Clean white background with light grid
plt.rcParams['font.family'] = 'Arial'  # Use Arial font like MATLAB

# Set up 4x4 subplot grid
fig, axes = plt.subplots(4, 4, figsize=(18, 16), sharex=True, sharey=True)
axes = axes.flatten()

# Get all unique site_ids (should be 14)
site_ids = sorted(df_GR['site_id'].unique())

# Define bin edges
bin_edges = np.arange(-4, 5, 1)

# Loop through each site_id and plot
for i, site_id in enumerate(site_ids):
    ax = axes[i]
    df_site = df_GR[df_GR['site_id'] == site_id]
    diff = df_site['wspeed_nwp'] - df_site['wspeed_pred']
    # Get histogram data 
    counts, bins = np.histogram(diff, bins=bin_edges)
    total_count = len(diff)
    percentages = (counts/total_count)*100 
    # Plot using bar
    ax.bar(
        bin_edges[:-1], percentages, 
        width=np.diff(bin_edges)[0], 
        align='edge', alpha=0.7,
        edgecolor='white')
    ax.yaxis.grid(True, linestyle='--', alpha=0.7)
    ax.set_xlim((-4, 4))
    ax.set_ylim((0, 80))
    ax.set_xticks(bin_edges)
    ax.tick_params(axis='x', labelbottom=True)
    short_name = locations.loc[locations['site_id'] == site_id, 'short_name'].values[0]
    ax.text(
        0.98, 0.98, f"site_id {site_id}\n{short_name}",
        transform=ax.transAxes,
        ha='right', va='top',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'),
        fontsize=10)

# Hide any unused subplots if site_ids < 16
for j in range(len(site_ids), 16):
    fig.delaxes(axes[j])

fig.suptitle("Distribution of wind speed discrepancies", fontsize=24, weight='bold')
fig.text(0.5, 0.91, r"$(V_{NWP} - V_{GR})$", ha='center', fontsize=18)
fig.text(0.5, 0.04, "Wind speed difference (m/s)", ha='center', fontsize=18)
fig.text(0.06, 0.5, "Percentage (%)", ha='center', va='center', rotation='vertical', fontsize=18)
# plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:

# Filter for June 1 to June 7 
start = pd.to_datetime('2025-06-01')
end = pd.to_datetime('2025-06-07 23:59:59')
df_june = df[(df['Datetime'] >= start) & (df['Datetime'] <= end)]

# Filter for site_id 111115 and June 1-7, 2025
site_id = 111115
start = pd.to_datetime('2025-06-01')
end = pd.to_datetime('2025-06-07 23:59:59')
df_site = df[df['site_id'] == site_id].copy()
df_site['Datetime'] = pd.to_datetime(df_site['time'])
df_june = df_site[(df_site['Datetime'] >= start) & (df_site['Datetime'] <= end)]
df_june = df_june.sort_values('Datetime')

In [ ]:
# Plot wind speed predictions vs NWP for site_id 111115
plt.figure(figsize=(12, 6))
plt.plot(df_june['Datetime'], df_june['wspeed_pred'], label='Predicted Wind Speed')
plt.plot(df_june['Datetime'], df_june['wspeed_nwp'], label='NWP Wind Speed')
plt.xlabel('Date')
plt.ylabel('Wind Speed (m/s)')
plt.title('Wind Speed: Prediction vs NWP (June 1-7, 2025) for site_id 111115')
plt.legend()
plt.show()

# # Plot wind direction predictions vs NWP for site_id 111115
# plt.figure(figsize=(12, 6))
# plt.plot(df_june['Datetime'], df_june['wdir_pred'], label='Predicted Wind Direction')
# plt.plot(df_june['Datetime'], df_june['wdir_nwp'], label='NWP Wind Direction')
# plt.xlabel('Date')
# plt.ylabel('Wind Direction (degrees)')
# plt.title('Wind Direction: Prediction vs NWP (June 1-7, 2025) for site_id 111115')
# plt.legend()
# plt.show()